In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('/content/EVSE-B-PowerCombined.csv')

# Drop irrelevant columns if needed
df = df[['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Label']]

# Convert labels to binary (0 for normal, 1 for attack)
df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)

# Standardize the features
scaler = StandardScaler()
features = scaler.fit_transform(df[['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']])
labels = df['Label'].values

# Train-validation-test split
X_train, X_temp, y_train, y_temp = train_test_split(features, labels, test_size=0.4, stratify=labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

# Define the ANN model
class EVChargingANN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(EVChargingANN, self).__init__()
        self.fc1 = torch.nn.Linear(input_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = 0.3

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.fc3(x)
        return torch.sigmoid(x).squeeze()

# Instantiate the model
model = EVChargingANN(input_dim=features.shape[1], hidden_dim=16, output_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

# Training function
def train():
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Evaluation function
def evaluate(loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch).squeeze()
            preds = (outputs > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(y_batch.cpu())
            all_probs.append(outputs.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = None
    if len(set(all_labels)) > 1:
        roc_auc = roc_auc_score(all_labels, all_probs)

    return accuracy, precision, f1, roc_auc

# Training loop with validation
for epoch in range(50):
    loss = train()
    val_accuracy, val_precision, val_f1, val_auc = evaluate(val_loader)
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Val Precision: {val_precision:.4f}, Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}')

# Test evaluation
test_accuracy, test_precision, test_f1, test_roc_auc = evaluate(test_loader)
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC: {test_roc_auc:.4f}')


Epoch 1, Loss: 0.3214, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.7553
Epoch 2, Loss: 0.2907, Val Accuracy: 0.8784, Val Precision: 0.8780, Val F1: 0.9350, Val AUC: 0.8392
Epoch 3, Loss: 0.2882, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.7532
Epoch 4, Loss: 0.2845, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.7572
Epoch 5, Loss: 0.2846, Val Accuracy: 0.8797, Val Precision: 0.8792, Val F1: 0.9357, Val AUC: 0.7810
Epoch 6, Loss: 0.2871, Val Accuracy: 0.8804, Val Precision: 0.8798, Val F1: 0.9361, Val AUC: 0.7644
Epoch 7, Loss: 0.2856, Val Accuracy: 0.8825, Val Precision: 0.8817, Val F1: 0.9371, Val AUC: 0.8010
Epoch 8, Loss: 0.2831, Val Accuracy: 0.8800, Val Precision: 0.8795, Val F1: 0.9359, Val AUC: 0.7711
Epoch 9, Loss: 0.2833, Val Accuracy: 0.8814, Val Precision: 0.8806, Val F1: 0.9365, Val AUC: 0.7893
Epoch 10, Loss: 0.2769, Val Accuracy: 0.8758, Val Precision: 0.8757, Val F1: 0.9337, Val AUC: 0.7656

In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader  # Updated import
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('/content/EVSE-B-PowerCombined.csv')

# Drop irrelevant columns if needed
df = df[['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Label']]
df

# Convert labels to binary (0 for normal, 1 for attack)
df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)
df

# Standardize the features
scaler = StandardScaler()
features = scaler.fit_transform(df[['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']])
features

# Create edge index (assuming each row is a node connected to the next for simplicity)
edge_index = torch.tensor([[i, i + 1] for i in range(len(df) - 1)], dtype=torch.long).t()

# Convert to PyTorch Geometric data format
x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(df['Label'].values, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
data

# Train-validation-test split
train_data, temp_data = train_test_split(list(range(len(df))), test_size=0.4, stratify=y)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=y[temp_data])

train_mask = torch.tensor([i in train_data for i in range(len(df))], dtype=torch.bool)
val_mask = torch.tensor([i in val_data for i in range(len(df))], dtype=torch.bool)
test_mask = torch.tensor([i in test_data for i in range(len(df))], dtype=torch.bool)

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

train_loader = DataLoader([data], batch_size=1, shuffle=True)
val_loader = DataLoader([data], batch_size=1)
test_loader = DataLoader([data], batch_size=1)

train_loader
val_loader
test_loader

class EVChargingGNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):  # Corrected __init__
        super(EVChargingGNN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = 0.3

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.fc(x)
        return torch.sigmoid(x).squeeze()

# Instantiate the model
model = EVChargingGNN(input_dim=x.shape[1], hidden_dim=16, output_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.BCELoss()

def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        output = model(data)[data.train_mask]
        loss = criterion(output, data.y[data.train_mask].float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate(loader, mask_type):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for data in loader:
            output = model(data)[getattr(data, f"{mask_type}_mask")]
            preds = (output > 0.5).float()  # Threshold at 0.5 for binary classification
            all_preds.append(preds.cpu())
            all_labels.append(data.y[getattr(data, f"{mask_type}_mask")].cpu())
            all_probs.append(output.cpu())

    # Concatenate predictions, labels, and probabilities
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = None
    if len(set(all_labels)) > 1:
        roc_auc = roc_auc_score(all_labels, all_probs)

    return accuracy, precision, f1, roc_auc

# Training loop with validation
for epoch in range(50):
    loss = train()
    val_accuracy, val_precision, val_f1, val_auc = evaluate(val_loader, 'val')
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Val Precision: {val_precision:.4f}, Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}')

# Test evaluation
test_accuracy, test_precision, test_f1, test_roc_auc = evaluate(test_loader, 'test')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC: {test_roc_auc:.4f}')

